In [13]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from umap import UMAP

# Fonctions de Nettoyage
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Mise en minuscule
    text = text.lower()
    # Suppression des caractères non-ASCII (garde les accents courants si nécessaire, sinon strict)
    text = re.sub(r'[^\w\s\-\'àâçéèêëîïôûùüÿñæœ]', '', text)
    # Normalisation des espaces (pas de double espace)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = pd.read_csv('../large_idiom_set_fr_eng.csv')

# Application du nettoyage
df['clean_french'] = df['original_text'].apply(clean_text)
df['clean_english'] = df['text'].apply(clean_text)

# ===== NEW APPROACH: Use contains_idioms column =====
target_idiom_str = "aller droit à le but"

# Find all rows where the target idiom appears in contains_idioms
# contains_idioms likely contains a list or comma-separated string of idiom IDs
df_target = df[df['contains_idioms'].astype(str).str.contains(target_idiom_str, na=False)].copy()

print(f"Traitement de {len(df_target)} phrases contenant '{target_idiom_str}'")
print(f"Idiom entries: {df_target['contains_idioms'].unique()[:5]}")  # Show sample idiom IDs

# Reset index to avoid misalignment
df_target = df_target.reset_index(drop=True)

# Génération des Variations
idiom_words = clean_text(target_idiom_str).split()

# Création des colonnes de variations
variations_dict = {'original': df_target['clean_french'].values}

for word in idiom_words:
    # On retire le mot proprement avec regex (\b pour word boundary)
    col_name = f'deleted_{word}'
    variations_dict[col_name] = df_target['clean_french'].str.replace(
        rf'\b{word}\b', '', regex=True
    ).apply(clean_text).values

df_variations = pd.DataFrame(variations_dict)

# Vérification 
print("\n--- Vérification des variations ---")
print(f"Original text          : {df_variations['original'].iloc[0]}")

for word in idiom_words:
    col_name = f'deleted_{word}'
    print(f"Word '{word}' deleted : {df_variations[col_name].iloc[0]}")

# Show sample rows
print("\n--- Sample rows (first 3) ---")
print(df_variations.head(3).to_string())

# Save variations to CSV
df_variations.to_csv('idiom_variations.csv', index=False)
print(f"\nVariations saved to 'idiom_variations.csv'")
print(f"Total variations generated: {len(df_variations)} rows × {len(df_variations.columns)} columns")

Traitement de 101 phrases contenant 'aller droit à le but'
Idiom entries: ['aller droit à le but' 'aller droit à le but,en avoir marre']

--- Vérification des variations ---
Original text          : et maintenant une jeune femme qui va droit au but ma sœur roberta tubbs
Word 'aller' deleted : et maintenant une jeune femme qui va droit au but ma sœur roberta tubbs
Word 'droit' deleted : et maintenant une jeune femme qui va au but ma sœur roberta tubbs
Word 'à' deleted : et maintenant une jeune femme qui va droit au but ma sœur roberta tubbs
Word 'le' deleted : et maintenant une jeune femme qui va droit au but ma sœur roberta tubbs
Word 'but' deleted : et maintenant une jeune femme qui va droit au ma sœur roberta tubbs

--- Sample rows (first 3) ---
                                                                  original                                                            deleted_aller                                                      deleted_droit                            

In [15]:
MODEL_NAME = 'BAAI/bge-m3'

# --- Encodage avec BGE-M3 ---
print(f"\nChargement du modèle {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

# Stockage structuré pour l'analyse
data_records = []

print("Encodage des références anglaises...")
eng_embeddings = model.encode(df_target['clean_english'].tolist(), show_progress_bar=True)

for idx, (text, emb) in enumerate(zip(df_target['clean_english'], eng_embeddings)):
    data_records.append({
        'idiom_id': idx,
        'text': text,
        'type': 'english_ref',
        'variation': 'english',
        'embedding': emb
    })

# Encoder chaque variation française
for var_name, texts in variations_dict.items():
    print(f"Encodage de la variation : {var_name}...")
    # BGE-M3 est plus lourd, la barre de progression est utile
    embeddings = model.encode(texts.tolist(), show_progress_bar=True)
    
    for idx, (text, emb) in enumerate(zip(texts, embeddings)):
        # Calculer la Distance Cosinus avec la version anglaise correspondante
        sim = cosine_similarity([emb], [eng_embeddings[idx]])[0][0]
        
        data_records.append({
            'idiom_id': idx,
            'text': text,
            'type': 'french_variation',
            'variation': var_name,
            'embedding': emb,
            'cosine_distance_to_eng': sim
        })

df_results = pd.DataFrame(data_records)

# Analyse de l'Influence par Distance Cosinus
print("\n=== Influence des mots (Basée sur la Distance Cosinus avec BGE-M3) ===")
print("Plus la distance augmente par rapport à l'original, plus le mot supprimé était important.")

if not df_results.empty:
    base_dist = df_results[df_results['variation'] == 'original']['cosine_distance_to_eng'].mean()
    print(f"Similarité Moyenne (Original FR <-> EN) : {base_dist:.4f}")

    for word in idiom_words:
        var_name = f'deleted_{word}'
        var_dist = df_results[df_results['variation'] == var_name]['cosine_distance_to_eng'].mean()
        impact = var_dist - base_dist
        print(f"Sans '{word}': Dist = {var_dist:.4f} (Impact: {impact:.4f})")

    # Projection UMAP (Metric='cosine') ---
    print("\nProjection UMAP avec métrique Cosinus...")
    all_embeddings = np.stack(df_results['embedding'].values)

    reducer = UMAP(n_components=2, metric='cosine', random_state=42)
    projections = reducer.fit_transform(all_embeddings)

    df_results['x'] = projections[:, 0]
    df_results['y'] = projections[:, 1]

    # Renommage pour compatibilité standard
    export_df = df_results.drop(columns=['embedding']).copy()

    df_results.to_parquet('idiom_analysis_cosine_bgem3.parquet')
    export_df.to_csv('idiom_analysis_viz_bgem3.csv', index=False)
    print("\nFichiers sauvegardés : 'idiom_analysis_cosine_bgem3.parquet' et 'idiom_analysis_viz_bgem3.csv'")
else:
    print("Aucune donnée à analyser.")



Chargement du modèle BAAI/bge-m3...
Encodage des références anglaises...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Encodage de la variation : original...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Encodage de la variation : deleted_aller...


...

Encodage de la variation : deleted_but...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


=== Influence des mots (Basée sur la Distance Cosinus avec BGE-M3) ===
Plus la distance augmente par rapport à l'original, plus le mot supprimé était important.
Similarité Moyenne (Original FR <-> EN) : 0.7405
Sans 'aller': Dist = 0.7342 (Impact: -0.0063)
Sans 'droit': Dist = 0.7209 (Impact: -0.0196)
Sans 'à': Dist = 0.7401 (Impact: -0.0004)
Sans 'le': Dist = 0.7401 (Impact: -0.0004)
Sans 'but': Dist = 0.7236 (Impact: -0.0169)

Projection UMAP avec métrique Cosinus...


C:\Users\benja\AppData\Roaming\Python\Python312\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



Fichiers sauvegardés : 'idiom_analysis_cosine_bgem3.parquet' et 'idiom_analysis_viz_bgem3.csv'
